# Global Superstore Data Preprocessing
This notebook performs basic data cleaning and preprocessing

## Import Libraries

In [1]:
import pandas as pd
import numpy as np

## Load Data

In [2]:
# Load the data (with encoding handling)
df = pd.read_csv('Global_Superstore.csv', encoding='latin-1')

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print("\nColumn names and types:")
print(df.dtypes)

Total rows: 51290
Total columns: 24

Column names and types:
Row ID              int64
Order ID           object
Order Date         object
Ship Date          object
Ship Mode          object
Customer ID        object
Customer Name      object
Segment            object
City               object
State              object
Country            object
Postal Code       float64
Market             object
Region             object
Product ID         object
Category           object
Sub-Category       object
Product Name       object
Sales             float64
Quantity            int64
Discount          float64
Profit            float64
Shipping Cost     float64
Order Priority     object
dtype: object


## Check for Missing Values

In [3]:
print("\nMISSING VALUES CHECK")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("No missing values found!")


MISSING VALUES CHECK
Postal Code    41296
dtype: int64


## Handle Missing Postal Codes

In [4]:
if df['Postal Code'].isnull().sum() > 0:
    print(f"\nPostal Code has {df['Postal Code'].isnull().sum()} missing values")
    print("Filling with 'Unknown'")
    df['Postal Code'] = df['Postal Code'].fillna('Unknown')


Postal Code has 41296 missing values
Filling with 'Unknown'


## Convert Date Columns

In [5]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')


## Create Derived Columns

In [6]:

# Shipping time in days
df['Shipping_Days'] = (df['Ship Date'] - df['Order Date']).dt.days

# Extract year, month, quarter
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.month
df['Order_Quarter'] = df['Order Date'].dt.quarter

# Profit margin percentage
df['Profit_Margin'] = (df['Profit'] / df['Sales']) * 100

# Profit category
df['Profit_Category'] = pd.cut(df['Profit'], 
                                bins=[-np.inf, 0, 100, 500, np.inf],
                                labels=['Loss', 'Low Profit', 'Medium Profit', 'High Profit'])


## Remove Duplicates

In [7]:
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()


Duplicate rows found: 0


## Outlier Detection

In [8]:

def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers)

for col in ['Sales', 'Profit', 'Quantity', 'Discount']:
    outlier_count = detect_outliers_iqr(df, col)
    print(f"{col}: {outlier_count} outliers detected")

Sales: 5655 outliers detected
Profit: 9755 outliers detected
Quantity: 877 outliers detected
Discount: 4172 outliers detected


## Data Validation

In [9]:

# Check for negative quantities
negative_qty = df[df['Quantity'] < 0]
print(f"Negative quantities: {len(negative_qty)}")

# Check for invalid discounts (should be between 0 and 1)
invalid_discount = df[(df['Discount'] < 0) | (df['Discount'] > 1)]
print(f"Invalid discounts: {len(invalid_discount)}")

Negative quantities: 0
Invalid discounts: 0


## Save Preprocessed Data

In [10]:
df.to_csv('Global_Superstore_Preprocessed.csv', index=False)


## Summary Statistics

In [11]:
print("\nNumerical columns summary:")
print(df[['Sales', 'Quantity', 'Discount', 'Profit', 'Shipping Cost']].describe())

print("PREPROCESSING COMPLETE!")
print(f"Final dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Numerical columns summary:
              Sales      Quantity      Discount        Profit  Shipping Cost
count  51290.000000  51290.000000  51290.000000  51290.000000   51290.000000
mean     246.490581      3.476545      0.142908     28.610982      26.375915
std      487.565361      2.278766      0.212280    174.340972      57.296804
min        0.444000      1.000000      0.000000  -6599.978000       0.000000
25%       30.758625      2.000000      0.000000      0.000000       2.610000
50%       85.053000      3.000000      0.000000      9.240000       7.790000
75%      251.053200      5.000000      0.200000     36.810000      24.450000
max    22638.480000     14.000000      0.850000   8399.976000     933.570000
PREPROCESSING COMPLETE!
Final dataset shape: (51290, 30)
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country', 'Postal Code', 'Market', 'Region', 'Product ID', 'Category', 'Sub-Category', 'P